[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/06_multihead_attention.ipynb)

# 🔴 Hard: Multi-Head Attention

Implement **Multi-Head Attention** from scratch — the core building block of the Transformer.

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h) W^O$$
$$\text{head}_i = \text{Attention}(Q W_i^Q,\; K W_i^K,\; V W_i^V)$$

### Signature
```python
class MultiHeadAttention:
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, Q, K, V) -> torch.Tensor: ...
```

### Requirements
- Use `nn.Linear(d_model, d_model)` for `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`
- `d_k = d_model // num_heads` per head
- `forward(Q, K, V)`: Q is `(B, seq_q, d_model)`, K/V are `(B, seq_k, d_model)`
- Must support **cross-attention** (`seq_q != seq_k`)
- Do **NOT** use `torch.nn.MultiheadAttention`
- You **may** use `torch.softmax` and `torch.matmul`

### Steps
1. Project: `q = self.W_q(Q)`, `k = self.W_k(K)`, `v = self.W_v(V)`
2. Reshape to `(B, num_heads, seq, d_k)`
3. Scaled dot-product attention per head
4. Concat heads → `(B, seq_q, d_model)`
5. Output projection: `self.W_o(concat)`

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import torch.nn as nn
import math

In [25]:
# ✏️ YOUR IMPLEMENTATION HERE

class MultiHeadAttention:
    def __init__(self, d_model: int, num_heads: int):
        # pass  # Initialize W_q, W_k, W_v, W_o
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, Q, K, V):
        # pass  # Implement multi-head attention
        batch_size, seq_len_q = Q.size(0), Q.size(1)
        seq_len_kv = K.size(1)

        Q = self.W_q(Q)
        K = self.W_k(K)
        V = self.W_v(V)
        
        print(Q.shape, K.shape, V.shape)
        
        # [Batch, Seq_len, d_model] -> [Batch, Seq_len, num_heads, head_dim]
        Q_split = Q.view(batch_size, seq_len_q, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        K_split = K.view(batch_size, seq_len_kv, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        V_split = V.view(batch_size, seq_len_kv, self.num_heads, self.head_dim).permute(0, 2, 1, 3)

        print(Q_split.shape, K_split.shape, V_split.shape)

        multi_head_attn = torch.softmax(Q_split @ K_split.transpose(-2, -1) / math.sqrt(self.head_dim), dim=-1) @ V_split
        print(multi_head_attn.shape) # the seq length of the output is the same as the query sequence length
        attn = multi_head_attn.permute(0, 2, 1, 3).contiguous().view(batch_size, seq_len_q, self.d_model) # concatenate the heads
        print(attn.shape)
        return self.W_o(attn)

In [26]:
# 🧪 Debug
torch.manual_seed(0)
mha = MultiHeadAttention(d_model=32, num_heads=4)
print("W_q type:", type(mha.W_q))          # should be nn.Linear
print("W_q.weight shape:", mha.W_q.weight.shape)  # (32, 32)

x = torch.randn(2, 6, 32)
out = mha.forward(x, x, x)
print("Output shape:", out.shape)          # (2, 6, 32)

# Cross-attention
Q = torch.randn(1, 3, 32)
K = torch.randn(1, 7, 32)
V = torch.randn(1, 7, 32)
out2 = mha.forward(Q, K, V)
print("Cross-attn shape:", out2.shape)     # (1, 3, 32)

W_q type: <class 'torch.nn.modules.linear.Linear'>
W_q.weight shape: torch.Size([32, 32])
torch.Size([2, 6, 32]) torch.Size([2, 6, 32]) torch.Size([2, 6, 32])
torch.Size([2, 4, 6, 8]) torch.Size([2, 4, 6, 8]) torch.Size([2, 4, 6, 8])
torch.Size([2, 4, 6, 8])
torch.Size([2, 6, 32])
Output shape: torch.Size([2, 6, 32])
torch.Size([1, 3, 32]) torch.Size([1, 7, 32]) torch.Size([1, 7, 32])
torch.Size([1, 4, 3, 8]) torch.Size([1, 4, 7, 8]) torch.Size([1, 4, 7, 8])
torch.Size([1, 4, 3, 8])
torch.Size([1, 3, 32])
Cross-attn shape: torch.Size([1, 3, 32])


In [20]:
# ✅ SUBMIT
from torch_judge import check
check("mha")


🧪 Testing: Multi-Head Attention (Hard)
──────────────────────────────────────────────────
torch.Size([2, 6, 32]) torch.Size([2, 6, 32]) torch.Size([2, 6, 32])
torch.Size([2, 4, 6, 8]) torch.Size([2, 4, 6, 8]) torch.Size([2, 4, 6, 8])
torch.Size([2, 4, 6, 8])
torch.Size([2, 6, 32])
  ✅ [1/6] Output shape (1.7ms)
  ✅ [2/6] Uses nn.Linear with correct shapes (0.2ms)
torch.Size([1, 4, 16]) torch.Size([1, 4, 16]) torch.Size([1, 4, 16])
torch.Size([1, 2, 4, 8]) torch.Size([1, 2, 4, 8]) torch.Size([1, 2, 4, 8])
torch.Size([1, 2, 4, 8])
torch.Size([1, 4, 16])
  ✅ [3/6] Numerical correctness vs reference (0.7ms)
torch.Size([1, 4, 16]) torch.Size([1, 4, 16]) torch.Size([1, 4, 16])
torch.Size([1, 2, 4, 8]) torch.Size([1, 2, 4, 8]) torch.Size([1, 2, 4, 8])
torch.Size([1, 2, 4, 8])
torch.Size([1, 4, 16])
  ✅ [4/6] Gradient flow (0.7ms)
torch.Size([1, 3, 32]) torch.Size([1, 7, 32]) torch.Size([1, 7, 32])
torch.Size([1, 4, 3, 8]) torch.Size([1, 4, 7, 8]) torch.Size([1, 4, 7, 8])
torch.Size([1, 4, 3,